**[🏠 Course Home](00_START_HERE.ipynb)** | **Sheet 1 of 4: Foundations** | [➡️ Next: Sheet 2 (Quadratic Approx)](02_quadratic_laplace_approximation.ipynb)

---

# Estimating Population Mean Height: Frequentist vs. Bayesian Grid Approximation

This notebook provides a rigorous, step-by-step pedagogical walkthrough of estimating a population mean height ($\mu$) and standard deviation ($\sigma$) from data using both **Classical Frequentist** inference and **Bayesian Grid Approximation** in **R**.

---

## Table of Contents
1. **Part 1: The Problem & The Data** (Generative Model & Simulation)
2. **Part 2: The Frequentist Paradigm** (Point Estimates, Standard Error, $t$-based Confidence Intervals)
3. **Part 3: The Bayesian Paradigm — Step-by-Step Foundations & 7 Deep Dives**
   - Step 0: Setting Priors & Prior Predictive Simulation
   - Deep Dive 1: Why Likelihood is the Product of `dnorm()`
   - Deep Dive 2: Why Posterior $\propto$ Likelihood $\times$ Prior
   - Deep Dive 3: How & Why We Compute the Log-Prior
   - Deep Dive 4: The Log-Sum-Exp Trick & Numerical Underflow
   - Deep Dive 5: Why We Sample from the 2D Grid
   - Deep Dive 6: Summarizing the Posterior Distribution
   - Deep Dive 7: The Posterior Predictive Distribution (Parameters vs. Observables)
4. **Part 4: Method 3A: Grid Approximation (The Code Pipeline)**
5. **Part 5: Visual Comparisons** (Parameter Uncertainty vs. Outcome Prediction)
6. **Part 6: Prior Sensitivity & Shrinkage Analysis** (Flat vs. Weak vs. Skeptical Priors)
7. **Part 7: Common Traps & FAQs** (The Bernstein-von Mises Theorem & Natural $\sqrt{n}$ Convergence)
8. **Part 8: Hands-On Challenge Exercises**

## Part 1: The Problem & The Data

### 1.1 The Generative Model
Suppose we want to know the average height of an adult population. We collect a sample of $n = 50$ adult heights: $y = (y_1, y_2, \dots, y_n)$.

We assume heights follow a Gaussian (Normal) generative process:
$$y_i \sim \text{Normal}(\mu, \sigma^2)$$
where:
- $\mu$ is the **true population mean height** (in cm).
- $\sigma$ is the **true standard deviation** (spread of heights among individuals, in cm).

In [ ]:
# Set seed for reproducibility (comment out to see random sampling fluctuations!)
set.seed(42)
n <- 50
true_mu <- 172.5
true_sigma <- 8.0

# Simulate sample data
heights <- rnorm(n, mean = true_mu, sd = true_sigma)

cat(sprintf("Sample Size (n): %d\n", n))
cat(sprintf("Sample Mean:     %.2f cm\n", mean(heights)))
cat(sprintf("Sample SD:       %.2f cm\n", sd(heights)))

# Visual Inspection of Sample
hist(heights, breaks = 12, col = "lightblue", border = "white", 
     main = "Observed Sample of 50 Adult Heights", xlab = "Height (cm)", las = 1)
abline(v = mean(heights), col = "red", lwd = 2, lty = 2)
legend("topright", legend = c("Sample Mean"), col = "red", lty = 2, lwd = 2, bty = "n")

## Part 2: The Frequentist Paradigm

### 2.1 Philosophy
- The parameter $\mu$ is a **fixed, single, unknown constant**.
- Probability is defined strictly as **long-run relative frequency** under infinite hypothetical repetitions of the sampling experiment.

### 2.2 Point Estimation & Standard Error
- **Point Estimate (Sample Mean)**: $\hat{\mu} = \bar{y} = \frac{1}{n} \sum_{i=1}^n y_i$
- **Standard Error of the Mean**: $\text{SE} = \frac{s}{\sqrt{n}}$, where $s = \sqrt{\frac{1}{n-1}\sum (y_i - \bar{y})^2}$.
- **95% Confidence Interval (CI)**:
  $$\text{CI}_{95\%} = \left[ \bar{y} - t_{n-1, 0.975} \cdot \text{SE}, \; \bar{y} + t_{n-1, 0.975} \cdot \text{SE} \right]$$

> **Crucial Frequentist Interpretation**: A 95% Confidence Interval does **not** mean there is a 95% chance $\mu$ is inside this interval. The true parameter $\mu$ is fixed (it is either in the interval or not, 0 or 1). Instead, it means: *If we repeat this sampling experiment infinitely many times and compute a CI each time, 95% of those calculated intervals will capture $\mu$.*

In [ ]:
# Frequentist estimation using t.test
freq_fit  <- t.test(heights)
freq_mean <- freq_fit$estimate
freq_ci   <- freq_fit$conf.int
freq_se   <- sd(heights) / sqrt(n)

cat("=== Frequentist Results ===\n")
cat(sprintf("Point Estimate (Sample Mean): %.2f cm\n", freq_mean))
cat(sprintf("Standard Error (SE):          %.3f cm\n", freq_se))
cat(sprintf("95%% Confidence Interval:       [%.2f, %.2f] cm\n", freq_ci[1], freq_ci[2]))

## Part 3: The Bayesian Paradigm — Step-by-Step Foundations & 7 Deep Dives

### 3.1 Philosophy
- The parameter $\mu$ is treated as a **random variable** reflecting our state of knowledge/epistemic uncertainty.
- Probability represents a **quantifiable degree of belief**, updated as new evidence is observed.

---

### Step 0: Setting Priors & Prior Predictive Simulation
Before analyzing our 50 data points, we establish our prior distributions based on domain knowledge of human biology:
- $\mu \sim \text{Normal}(\mu_0 = 170, \sigma_0 = 15)$: We expect adult population averages to hover around 170 cm; 95% of our prior mass spans $[140, 200]$ cm.
- $\sigma \sim \text{Uniform}(0, 30)$: Variation is strictly positive, but population SD rarely exceeds 30 cm.

**Prior Predictive Check**: What kind of human heights does our model expect *before seeing any data*?

In [ ]:
# Prior Predictive Simulation
n_sim <- 1e4
prior_mu_sim    <- rnorm(n_sim, mean = 170, sd = 15)
prior_sigma_sim <- runif(n_sim, min = 0, max = 30)
prior_heights   <- rnorm(n_sim, mean = prior_mu_sim, sd = prior_sigma_sim)

cat(sprintf("Prior Predictive 95%% Interval for Individuals: [%.1f, %.1f] cm\n", 
            quantile(prior_heights, 0.025), quantile(prior_heights, 0.975)))

hist(prior_heights, breaks = 30, col = "lightgray", border = "white", 
     main = "Prior Predictive Distribution of Human Heights", xlab = "Height (cm)", las = 1)

### Deep Dive 1: Why Likelihood is the Product of `dnorm()`

Why does $P(y \mid \mu, \sigma) = \prod_{i=1}^n \text{dnorm}(y_i, \mu, \sigma)$?

1. **Conditional Independence (i.i.d.)**: Given the true parameters $(\mu, \sigma)$, knowing one person's height gives zero additional information about another person's height.
2. **Multiplication Rule for Joint Probability**: For independent events, the joint probability is the product of individual probabilities: $P(A \cap B) = P(A) \times P(B)$.
   $$P(y_1, y_2, \dots, y_n \mid \mu, \sigma) = P(y_1 \mid \mu, \sigma) \times P(y_2 \mid \mu, \sigma) \times \dots \times P(y_n \mid \mu, \sigma) = \prod_{i=1}^n P(y_i \mid \mu, \sigma)$$
3. **What `dnorm()` Evaluates**: For each individual measurement $y_i$, `dnorm()` evaluates the Gaussian probability density function:
   $$P(y_i \mid \mu, \sigma) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left( -\frac{(y_i - \mu)^2}{2\sigma^2} \right)$$
4. **Log-Sum Transformation in Code**: Multiplying 50 small decimals causes **arithmetic underflow** to `0.0` on a computer. Since $\log(a \cdot b) = \log a + \log b$, the product becomes a sum:
   $$\log P(y \mid \mu, \sigma) = \sum_{i=1}^n \log \text{dnorm}(y_i, \mu, \sigma) \quad \Longrightarrow \quad \text{R code: } \texttt{sum(dnorm(heights, mean=mu, sd=sigma, log=TRUE))}$$

---

### Deep Dive 2: Why Posterior $\propto$ Likelihood $\times$ Prior

1. **Derivation from Conditional Probability**:
   $$P(A \mid B) = \frac{P(A \cap B)}{P(B)} \quad \text{and} \quad P(B \mid A) = \frac{P(A \cap B)}{P(A)} \implies P(A \cap B) = P(B \mid A)P(A)$$
   $$P(A \mid B) = \frac{P(B \mid A)P(A)}{P(B)}$$
   Letting $A = (\mu, \sigma)$ (parameters) and $B = y$ (data):
   $$P(\mu, \sigma \mid y) = \frac{P(y \mid \mu, \sigma) P(\mu, \sigma)}{P(y)} = \frac{\text{Likelihood} \times \text{Prior}}{\text{Evidence}}$$

2. **Why the Denominator $P(y)$ is Just a Constant Normalizer**:
   $P(y) = \iint P(y \mid \mu, \sigma) P(\mu, \sigma) \, d\mu d\sigma = \sum_{\text{grid rows}} \text{Likelihood} \times \text{Prior}$.
   $P(y)$ evaluates to a single fixed number (a constant) with no free parameters. Therefore, the shape and relative probabilities of the posterior are strictly governed by:
   $$\text{Posterior} \propto \text{Likelihood} \times \text{Prior}$$

3. **Intuition: Data as a Plausibility Filter**:
   - $\text{Prior}$: Initial plausibility assigned to each $(\mu, \sigma)$ candidate before looking at data.
   - $\text{Likelihood}$: How well that candidate accounts for the observed 50 heights.
   - $\text{Multiplication}$: Reweights the prior—candidates with either low prior plausibility or poor likelihood are filtered out.

---

### Deep Dive 3: How & Why We Compute the Log-Prior

In the code, we write:
```r
log_prior_mu    <- dnorm(grid$mu, mean = 170, sd = 15, log = TRUE)
log_prior_sigma <- dunif(grid$sigma, min = 0, max = 30, log = TRUE)
log_prior       <- log_prior_mu + log_prior_sigma
```
1. **Independence of Priors**: Assuming $\mu$ and $\sigma$ are independent prior to seeing data:
   $$P(\mu, \sigma) = P(\mu) \times P(\sigma) \implies \log P(\mu, \sigma) = \log P(\mu) + \log P(\sigma)$$
2. **Scale Compatibility**: The log-likelihood produces values around $-174$. To add them via $\log(\text{Posterior}) = \log(\text{Likelihood}) + \log(\text{Prior})$, the prior must be computed in log units.

---

### Deep Dive 4: The Log-Sum-Exp Trick & Numerical Underflow

To normalize unnormalized log-posterior values $\log P_i$ into probabilities that sum to 1:
$$P_i = \frac{\exp(\log P_i)}{\sum_j \exp(\log P_j)}$$
- If $\log P_i \approx -174$, standard $\exp(-174)$ produces `0.0` (IEEE-754 underflow), yielding `0 / 0 = NaN`.
- **The Exact Mathematical Fix** (Log-Sum-Exp Trick): Factor out the maximum value $M = \max(\log P)$:
  $$P_i = \frac{\exp(\log P_i - M) \cdot \exp(M)}{\sum_j \exp(\log P_j - M) \cdot \exp(M)} = \frac{\exp(\log P_i - M)}{\sum_j \exp(\log P_j - M)}$$
- This shifts the highest log-probability to $\exp(0) = 1.0$, guaranteeing numerical stability with **zero mathematical distortion**:
  ```r
  post_prob <- exp(log_post - max(log_post))
  post_prob <- post_prob / sum(post_prob)
  ```

---

### Deep Dive 5: Why We Sample from the Posterior Grid

Instead of doing calculus or weighted matrix sums on the 90,000 grid points, we draw 10,000 weighted samples (`sample(..., prob = post_prob)`):
1. **Automatic Marginalization**: To isolate parameter $\mu$ and ignore $\sigma$, you simply look at `posterior_mu` without doing any integrals.
2. **Trivial Transformations**: To find the distribution of any non-linear function (e.g., Coefficient of Variation $\text{CV} = \sigma / \mu$), just compute `posterior_sigma / posterior_mu` directly on the sample vectors.
3. **Universal Paradigm**: Prepares you for MCMC (Stan / `brms`), which always operates on posterior sample vectors.

---

### Deep Dive 6: Summarizing the Posterior Distribution

- **Posterior Mean** (`mean(posterior_mu)`): Optimal point estimate under quadratic ($L_2$) loss.
- **Posterior Median** (`median(posterior_mu)`): Optimal point estimate under absolute ($L_1$) loss.
- **Posterior Standard Deviation** (`sd(posterior_mu)`): Bayesian measure of parameter uncertainty.
- **95% Credible Interval** (`quantile(posterior_mu, c(0.025, 0.975))`): Direct probability statement:
  > *"Given our prior and the observed data, there is a 95% probability that the population mean height $\mu$ lies between these bounds."*

---

### Deep Dive 7: The Posterior Predictive Distribution (Parameters vs. Observables)

The **Posterior Predictive Distribution (PPD)** answers: *"Now that we have learned from our sample of 50 people, what should we expect the **next individual person's height** to be?"*

#### 1. The Two Layers of Uncertainty:
Total predictive uncertainty is the sum of two distinct sources:
$$\text{Total Uncertainty} = \underbrace{\text{Parameter Uncertainty (Epistemic)}}_{\text{Shrinks to } 0 \text{ as } n \to \infty} + \underbrace{\text{Natural Biological Variation (Aleatoric)}}_{\text{Never shrinks to } 0}$$

- **Parameter Uncertainty** ($P(\mu, \sigma \mid y)$): We only measured 50 people, so the true population mean $\mu$ is uncertain (spread $\approx \pm 1.1\text{ cm}$).
- **Individual Variation** ($y_{\text{new}} \sim \text{Normal}(\mu, \sigma)$): Even if we knew $\mu$ and $\sigma$ with 100% precision, humans are not identical clones—individuals vary widely (spread $\approx \pm 8.0\text{ cm}$).

#### 2. The Mathematical Formulation:
We integrate the likelihood of a new individual across all plausible $(\mu, \sigma)$ parameter pairs, weighted by their posterior probabilities:
$$P(y_{\text{new}} \mid y) = \iint \underbrace{\text{Normal}(y_{\text{new}} \mid \mu, \sigma)}_{\text{Likelihood of new person}} \times \underbrace{P(\mu, \sigma \mid y)}_{\text{Posterior weight of parameters}} \, d\mu \, d\sigma$$

#### 3. How It Is Simulated in 1 Line of R Code:
```r
# Each simulated person draws from a slightly different (mu, sigma) pair from the posterior:
post_pred_heights <- rnorm(n = 1e4, mean = posterior_mu, sd = posterior_sigma)
```

#### 4. Numerical Comparison of Intervals:
- **95% Credible Interval for Population Mean $\mu$**: $\approx [170.3, 174.8]\text{ cm}$ (Width $\approx 4.5\text{ cm}$)
- **95% Prediction Interval for Individual Person $y_{\text{new}}$**: $\approx [156.4, 188.6]\text{ cm}$ (Width $\approx 32.2\text{ cm}$)

## Part 4: Method 3A: Grid Approximation (The Code Pipeline)

Now let us execute the complete Bayesian Grid Approximation workflow in R.

In [ ]:
# ==============================================================================
# 1. CREATE 2D PARAMETER GRID
# ==============================================================================
mu_grid    <- seq(from = 160, to = 185, length.out = 300)
sigma_grid <- seq(from = 2,   to = 20,  length.out = 300)
grid       <- expand.grid(mu = mu_grid, sigma = sigma_grid)

# ==============================================================================
# 2. COMPUTE LOG-LIKELIHOOD (Sum of log-densities)
# ==============================================================================
log_lik <- sapply(1:nrow(grid), function(i) {
  sum(dnorm(heights, mean = grid$mu[i], sd = grid$sigma[i], log = TRUE))
})

# ==============================================================================
# 3. COMPUTE LOG-PRIOR
# ==============================================================================
log_prior_mu    <- dnorm(grid$mu, mean = 170, sd = 15, log = TRUE)
log_prior_sigma <- dunif(grid$sigma, min = 0, max = 30, log = TRUE)
log_prior       <- log_prior_mu + log_prior_sigma

# ==============================================================================
# 4 & 5. UNNORMALIZED POSTERIOR & LOG-SUM-EXP NORMALIZATION
# ==============================================================================
log_post  <- log_lik + log_prior
post_prob <- exp(log_post - max(log_post))
post_prob <- post_prob / sum(post_prob)

# ==============================================================================
# 6. DRAW 10,000 SAMPLES FROM THE POSTERIOR
# ==============================================================================
samples_idx     <- sample(1:nrow(grid), size = 1e4, replace = TRUE, prob = post_prob)
posterior_mu    <- grid$mu[samples_idx]
posterior_sigma <- grid$sigma[samples_idx]

# ==============================================================================
# 7. SUMMARIZE POSTERIOR INFERENCE (POPULATION MEAN)
# ==============================================================================
bayes_mean <- mean(posterior_mu)
bayes_sd   <- sd(posterior_mu)
bayes_ci   <- quantile(posterior_mu, probs = c(0.025, 0.975))

# ==============================================================================
# 8. POSTERIOR PREDICTIVE SIMULATION (INDIVIDUAL HEIGHTS)
# ==============================================================================
post_pred_heights <- rnorm(1e4, mean = posterior_mu, sd = posterior_sigma)
pred_ci           <- quantile(post_pred_heights, probs = c(0.025, 0.975))

cat("=== Bayesian Inference Results ===\n")
cat(sprintf("Posterior Mean (mu):               %.2f cm\n", bayes_mean))
cat(sprintf("Posterior SD (mu):                 %.3f cm\n", bayes_sd))
cat(sprintf("95%% Credible Interval for Mean mu:  [%.2f, %.2f] cm\n", bayes_ci[1], bayes_ci[2]))
cat(sprintf("95%% Prediction Interval for Person: [%.2f, %.2f] cm\n", pred_ci[1], pred_ci[2]))

## Part 5: Visual Comparisons

Let us visualize:
1. **Parameter Estimation**: Frequentist Point Estimate & 95% CI vs. Bayesian Posterior Density & 95% Credible Interval.
2. **Outcome Prediction**: Uncertainty in the Mean ($\mu$) vs. Uncertainty in a New Individual ($y_{\text{new}}$).

In [ ]:
par(mfrow = c(1, 2))

# Plot 1: Estimating Population Mean (mu)
dens_mu <- density(posterior_mu)
plot(dens_mu, main = "Estimating Population Mean (mu)", 
     xlab = "Mean Height (cm)", col = "darkblue", lwd = 3, las = 1)
polygon(dens_mu, col = rgb(0, 0, 0.8, 0.15), border = NA)

# Frequentist Mean & CI
abline(v = freq_mean, col = "darkred", lwd = 2, lty = 2)
arrows(freq_ci[1], max(dens_mu$y)*0.4, freq_ci[2], max(dens_mu$y)*0.4, 
       angle = 90, code = 3, length = 0.08, col = "darkred", lwd = 2)
text(freq_mean, max(dens_mu$y)*0.47, "Frequentist 95% CI", col = "darkred", cex = 0.85)

# Bayesian Credible Interval
arrows(bayes_ci[1], max(dens_mu$y)*0.2, bayes_ci[2], max(dens_mu$y)*0.2, 
       angle = 90, code = 3, length = 0.08, col = "darkblue", lwd = 2)
text(bayes_mean, max(dens_mu$y)*0.27, "Bayesian 95% Credible Interval", col = "darkblue", cex = 0.85)

legend("topright", legend = c("Bayesian Posterior", "Frequentist Estimate"),
       col = c("darkblue", "darkred"), lty = c(1, 2), lwd = 2, bty = "n", cex = 0.8)

# Plot 2: Mean Uncertainty vs. Individual Prediction
dens_pred <- density(post_pred_heights)
plot(dens_pred, main = "Mean Uncertainty vs. Individual Prediction", 
     xlab = "Height (cm)", col = "darkgreen", lwd = 2, las = 1, ylim = c(0, max(dens_mu$y)))
polygon(dens_pred, col = rgb(0, 0.6, 0, 0.15), border = NA)

lines(dens_mu, col = "darkblue", lwd = 3)
polygon(dens_mu, col = rgb(0, 0, 0.8, 0.25), border = NA)

arrows(pred_ci[1], 0.02, pred_ci[2], 0.02, angle = 90, code = 3, length = 0.08, col = "darkgreen", lwd = 2)
text(mean(post_pred_heights), 0.03, "95% Individual Prediction Interval", col = "darkgreen", cex = 0.85)

legend("topright", legend = c("Uncertainty in Mean (mu)", "Prediction for Individual (y_new)"),
       col = c("darkblue", "darkgreen"), lwd = c(3, 2), bty = "n", cex = 0.8)

par(mfrow = c(1, 1))

## Part 6: Prior Sensitivity & Shrinkage Analysis

Let us compare three distinct priors on the exact same 50 height observations:
1. **Flat Prior**: $\mu \sim \text{Normal}(170, 1000)$ (Completely non-informative)
2. **Weakly Informative Prior**: $\mu \sim \text{Normal}(170, 15)$ (Our standard model)
3. **Skeptical / Stubborn Prior**: $\mu \sim \text{Normal}(140, 2)$ (Strong prior centered far from data)

In [ ]:
# Compute posterior under Flat Prior
lp_flat <- log_lik + dnorm(grid$mu, 170, 1000, log = TRUE) + dunif(grid$sigma, 0, 30, log = TRUE)
p_flat  <- exp(lp_flat - max(lp_flat)); p_flat <- p_flat / sum(p_flat)
mu_flat <- grid$mu[sample(1:nrow(grid), 1e4, TRUE, p_flat)]

# Compute posterior under Skeptical Prior (mu ~ Normal(140, 2))
lp_skep <- log_lik + dnorm(grid$mu, 140, 2, log = TRUE) + dunif(grid$sigma, 0, 30, log = TRUE)
p_skep  <- exp(lp_skep - max(lp_skep)); p_skep <- p_skep / sum(p_skep)
mu_skep <- grid$mu[sample(1:nrow(grid), 1e4, TRUE, p_skep)]

# Plot Sensitivity Comparison
plot(density(mu_flat), col = "black", lty = 2, lwd = 2, las = 1,
     main = "Prior Sensitivity Analysis (n = 50)", xlab = "Mean Height mu (cm)", xlim = c(135, 185))
lines(density(posterior_mu), col = "darkblue", lwd = 3)
lines(density(mu_skep), col = "darkred", lwd = 2)

legend("topleft", legend = c("Flat Prior (mu ~ N(170, 1000))", 
                             "Weak Prior (mu ~ N(170, 15))", 
                             "Skeptical Prior (mu ~ N(140, 2))"),
       col = c("black", "darkblue", "darkred"), lty = c(2, 1, 1), lwd = c(2, 3, 2), bty = "n")

cat(sprintf("Flat Prior Posterior Mean:      %.2f cm\n", mean(mu_flat)))
cat(sprintf("Weak Prior Posterior Mean:      %.2f cm\n", mean(posterior_mu)))
cat(sprintf("Skeptical Prior Posterior Mean: %.2f cm (Shrunk towards 140 cm!)\n", mean(mu_skep)))

## Part 7: Common Traps & FAQs

### Q1: Why are the Frequentist CI and Bayesian Credible Interval nearly identical here?
- **The Bernstein-von Mises Theorem**: Under weakly informative priors and moderate sample sizes ($n=50$), the Likelihood dominates the prior. When likelihood is Gaussian, the Bayesian posterior and Frequentist sampling distribution are asymptotically identical.
- **The Difference is in the Meaning**: Frequentist CI = coverage rate over infinite repeated experiments. Bayesian CI = direct epistemic probability of $\mu$ given this dataset.

### Q2: Why do Frequentists divide by $\sqrt{n}$, but Bayesians don't?
- In Frequentist formulas, $\text{SE} = s / \sqrt{n}$ is derived analytically.
- In Bayesian inference, we multiply $n$ likelihood terms together. The sum of $n$ squared error terms $(y_i - \mu)^2$ inside the exponential naturally concentrates the posterior variance by a factor of $\frac{1}{n}$, giving standard deviation $\frac{\sigma}{\sqrt{n}}$ **automatically without anyone writing $\sqrt{n}$ in the code!**

## Part 8: Hands-On Challenge Exercises

### Exercise 1: Finding Probability of Extreme Height
Using our posterior predictive sample vector `post_pred_heights`, calculate the probability that a randomly chosen person in this population is taller than **$185\text{ cm}$** ($6'1''$).

```r
# YOUR CODE HERE:
# prob_tall <- mean(post_pred_heights > 185)
```

### Exercise 2: Sample Size Overwhelming a Stubborn Prior
If we had $n = 500$ people instead of $n = 50$, what would happen to the Skeptical Prior estimate? *(Hint: Data likelihood multiplies 500 times, completely overwhelming even a stubborn prior).* 

---

**[🏠 Course Home](00_START_HERE.ipynb)** | [➡️ Next: Sheet 2 (Quadratic Approximation)](02_quadratic_laplace_approximation.ipynb)